1) Loading data and Preprocessing

In [1]:
import mne
import numpy as np

filename = 'A01T.gdf'
raw = mne.io.read_raw_gdf(filename)

raw.info

Extracting GDF parameters from A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...


/opt/anaconda3/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


<Info | 8 non-empty values
 bads: []
 ch_names: EEG-Fz, EEG-0, EEG-1, EEG-2, EEG-3, EEG-4, EEG-5, EEG-C3, EEG-6, ...
 chs: 25 EEG
 custom_ref_applied: False
 highpass: 0.5 Hz
 lowpass: 100.0 Hz
 meas_date: 2005-01-17 12:00:00 UTC
 nchan: 25
 projs: []
 sfreq: 250.0 Hz
 subject_info: <subject_info | his_id: A01, sex: 0, last_name: X, birthday: 1983-01-17>
>

In [2]:
#Filter the raw signal with a band pass filter in 7-35 HZ
raw.load_data()
raw.filter(l_freq=7., h_freq=35., method='fir')

Reading 0 ... 672527  =      0.000 ...  2690.108 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 7 - 35 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 7.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 6.00 Hz)
- Upper passband edge: 35.00 Hz
- Upper transition bandwidth: 8.75 Hz (-6 dB cutoff frequency: 39.38 Hz)
- Filter length: 413 samples (1.652 s)



<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>

In [3]:
events , _ = mne.events_from_annotations(raw)

Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]


In [4]:
# Remove the EOG channels and pick only eeg channels
raw.info['bads'] += ['EOG-left', 'EOG-central', 'EOG-right']

picks = mne.pick_types(raw.info, meg=False, eog=False, eeg=True, stim=False, exclude='bads')
picks

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21])

In [5]:
#Extracts epochs of 3 s from dataset  for all 4 classes
tmin, tmax = 3., 6.
event_id = dict({'769':7, '770':8, '771':9, '772':10})

epocks = mne.Epochs(raw, events=events, event_id=event_id, tmin=tmin, tmax=tmax, picks=picks, baseline=None, preload=True)

Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 751 original time points ...
1 bad epochs dropped


In [6]:
#Changing labels from 7,8,9,10 -> 1,2,3,4
labels = epocks.events[:,2] - 6
labels.shape

(287,)

2) Feature Extraction using Wavelet Packet Decomposition (WPD)

In [7]:
data = epocks.get_data()
data.shape

(287, 22, 751)

3) Feature Selection using Common spatial Pattern (CSP)

4) Classification using Multi-Layer Perceptron (MLP)

5) Evaluation